In [ ]:
import csv
import io
import re
import time
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, LinearConstraint, Bounds

from sympy import symbols, lambdify, sympify
from sympy.parsing.sympy_parser import parse_expr

from common_libs import check_vector

## Nạp và tiền xử lý dữ liệu

### 1. Đọc và chuẩn hóa dữ liệu
- Sử dụng ba bộ dữ liệu lợi suất danh mục **FF10, FF48 và FF100**.
- Chuẩn hóa cột `Date` về kiểu thời gian, chuyển lợi suất sang kiểu số và sắp xếp theo tháng.
- Kiểm tra số lượng tài sản, dữ liệu thiếu, giá trị vô hạn, tháng trùng lặp và tháng bị thiếu.

### 2. Làm sạch dữ liệu
- Thiết lập `CLEAN=True`, `FILL="zero"`.
- Thay các mã dữ liệu thiếu `-99.99`, `-999` và giá trị `NaN` bằng `0`. (cái này có thể chọn ở cell code dưới - hiện tại bài đang chọn là fill bằng 0)
- Chuyển lợi suất từ phần trăm sang dạng thập phân bằng cách chia cho 100.

### 3. Xây dựng cửa sổ trượt (Rolling Window) (cho giai đoạn kiểm tra hiệu quả của mô hình với bài toán tối ưu danh mục đầu tư)
- **Huấn luyện (Train):** 60 tháng.
- **Kiểm thử (Test):** 12 tháng tiếp theo.
- **Giai đoạn kiểm thử:** 07/1976 – 06/2006.
- Dịch cửa sổ 12 tháng sau mỗi lần kiểm thử, tạo tối đa 30 cửa sổ cho mỗi bộ dữ liệu nếu dữ liệu đầy đủ.

### 4. Chuẩn bị đầu vào cho thuật toán
Tại mỗi cửa sổ cho bộ dữ liệu, ta có:

| Biến | Ý nghĩa |
|---|---|
| `R_train` | Ma trận lợi suất huấn luyện |
| `R_test` | Ma trận lợi suất kiểm thử |
| `mu` | Vectơ lợi suất trung bình trên tập Train |
| `Sigma` | Ma trận hiệp phương sai mẫu |
| `rho` | Lợi suất mục tiêu của danh mục phân bổ đều |
| `x_equal` | Vectơ trọng số phân bổ đều \(1/N\) |

**Kết quả:** Dữ liệu được tổ chức thành các danh sách `windows_ff10`, `windows_ff48`, `windows_ff100`. Mỗi phần tử chứa dữ liệu huấn luyện, kiểm thử và các tham số cần thiết để đưa vào thuật toán tối ưu danh mục đầu tư.

In [24]:
def read_french_csv(path, n_assets):
    """
    Trả về:
        DataFrame gồm Date (YYYYMM)
        và n_assets cột lợi suất (%).
    """
    path = Path(path)

    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as z:
            names = [
                name for name in z.namelist()
                if name.lower().endswith(".csv")
            ]

            if len(names) != 1:
                raise ValueError(
                    f"ZIP phải chứa đúng 1 CSV: {names}"
                )

            raw_bytes = z.read(names[0])
    else:
        raw_bytes = path.read_bytes()

    try:
        text = raw_bytes.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = raw_bytes.decode("latin-1")

    lines = text.splitlines()

    def is_month(s):
        return re.fullmatch(
            r"\s*\d{6}\s*", str(s)
        ) is not None

    for i in range(len(lines) - 1):
        header = lines[i].split(",")
        first_row = lines[i + 1].split(",")

        if not (
            len(header) == n_assets + 1
            and len(first_row) == n_assets + 1
            and is_month(first_row[0])
        ):
            continue

        rows = [lines[i]]
        j = i + 1

        while j < len(lines):
            fields = lines[j].split(",")

            if not (
                len(fields) == n_assets + 1
                and is_month(fields[0])
            ):
                break

            rows.append(lines[j])
            j += 1

        # Bỏ qua các khối không đủ 12 tháng
        if len(rows) < 13:
            continue

        df = pd.read_csv(
            io.StringIO("\n".join(rows)),
            skipinitialspace=True
        )

        return df

    raise ValueError(
        "Không tìm thấy bảng lợi suất tháng "
        f"gồm {n_assets} danh mục."
    )


def read_portfolio_file(path, n_assets):
    """
    Input:
        path: đường dẫn file
        n_assets: 10, 48 hoặc 100

    Output:
        DataFrame với lợi suất gốc (%).
    """
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in [".csv", ".zip"]:
        df = read_french_csv(path, n_assets)

    elif suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path)

    else:
        raise ValueError(
            f"Định dạng không hỗ trợ: {suffix}"
        )

    # Cột đầu tiên là thời gian
    df = df.rename(
        columns={df.columns[0]: "Date"}
    )

    if df.shape[1] != n_assets + 1:
        raise ValueError(
            f"Cần {n_assets} cột lợi suất, "
            f"nhưng file có {df.shape[1] - 1}."
        )

    return df


def parse_returns(df):
    """
    Chuẩn hóa:
      - Date thành datetime
      - Các cột lợi suất thành kiểu số
      - Sắp xếp tăng dần theo thời gian

    Không xử lý missing codes.
    Không chia cho 100.
    """
    data = df.copy()

    if pd.api.types.is_datetime64_any_dtype(data["Date"]):
        data["Date"] = pd.to_datetime(data["Date"])
    else:
        dates = (
            data["Date"]
            .astype("string")
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

        data["Date"] = pd.to_datetime(
            dates,
            format="%Y%m",
            errors="raise"
        )

    asset_cols = data.columns.drop("Date")

    data[asset_cols] = data[asset_cols].apply(
        pd.to_numeric,
        errors="coerce"
    )

    data = (
        data.sort_values("Date")
        .reset_index(drop=True)
    )

    if data["Date"].duplicated().any():
        raise ValueError("Có tháng trùng lặp.")

    return data


def clean_returns(df, clean=True, fill="nan"):
    """
    Xử lý mã thiếu dữ liệu của Ken French.

    clean=False:
        Không thay đổi lợi suất.

    clean=True, fill="nan":
        -99.99, -999 -> NaN

    clean=True, fill="zero":
        -99.99, -999 -> 0
        Các NaN có sẵn cũng -> 0

    Output:
        DataFrame mới, không sửa df gốc.
        Lợi suất vẫn ở dạng %.
    """
    data = df.copy()

    if not clean:
        return data

    if fill not in ("nan", "zero"):
        raise ValueError(
            "fill phải là 'nan' hoặc 'zero'."
        )

    asset_cols = data.columns.drop("Date")

    # Bước 1: nhận diện mã thiếu
    data[asset_cols] = data[asset_cols].replace(
        [-99.99, -999.0],
        np.nan
    )

    # Bước 2: chọn cách xử lý
    if fill == "zero":
        data[asset_cols] = (
            data[asset_cols].fillna(0.0)
        )

    return data


def convert_to_decimal(df):
    """
    Chuyển lợi suất từ % sang thập phân.

    2.5% -> 0.025
    """
    data = df.copy()
    asset_cols = data.columns.drop("Date")

    data[asset_cols] = data[asset_cols] / 100.0

    return data


def inspect_returns(df, name="Dataset", decimal=False):
    """
    Kiểm tra:
      - Kích thước dữ liệu
      - Giai đoạn thời gian
      - Missing codes
      - NaN và Inf
      - Tháng lịch bị thiếu

    decimal=False: dữ liệu đang ở dạng %
    decimal=True: dữ liệu đã chia 100
    """
    data = df.copy()
    asset_cols = data.columns.drop("Date")
    R = data[asset_cols].to_numpy(dtype=float)

    factor = 100.0 if decimal else 1.0

    n_missing_codes = int(
        np.isclose(R, -99.99 / factor).sum()
        + np.isclose(R, -999.0 / factor).sum()
    )

    dates = pd.PeriodIndex(
        data["Date"], freq="M"
    )

    expected_dates = pd.period_range(
        dates.min(),
        dates.max(),
        freq="M"
    )

    missing_months = expected_dates.difference(dates)

    report = {
        "dataset": name,
        "n_months": len(data),
        "n_assets": len(asset_cols),
        "start": str(dates.min()),
        "end": str(dates.max()),
        "missing_codes": n_missing_codes,
        "nan_cells": int(np.isnan(R).sum()),
        "inf_cells": int(np.isinf(R).sum()),
        "missing_months": len(missing_months)
    }

    print(f"\n========== {name} ==========")

    for key, value in report.items():
        print(f"{key}: {value}")

    return report

In [25]:
PATH_FF10 = "D:\A. Học tập đại học\Đồ án 2\Code\FF10_AVG_Value_Weighted_Returns_Monthly.xlsx"
PATH_FF48 = "D:\A. Học tập đại học\Đồ án 2\Code\FF48_AVG_Value_Weighted_Returns_Monthly.xlsx"
PATH_FF100 = "D:\A. Học tập đại học\Đồ án 2\Code\FF100_AVG_Value_Weighted_Returns_Monthly.xlsx"


CLEAN = True #true or false
FILL = "zero" #nan or zero

# CẤU HÌNH THỰC NGHIỆM

TRAIN_MONTHS = 60
TEST_MONTHS = 12

FIRST_TEST = "1976-07"
LAST_TEST = "2006-06"

<>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:3: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:2: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:3: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
C:\Users\DoKhu\AppData\Local\Temp\ipykernel_11720\2100679153.py:1: SyntaxWarning: 

In [26]:
# 1. Đọc dữ liệu gốc
ff10_raw = read_portfolio_file(PATH_FF10,n_assets=10)
# 2. Chuẩn hóa
ff10_parsed = parse_returns(ff10_raw)
# 3. Kiểm tra dữ liệu gốc
ff10_report_raw = inspect_returns(ff10_parsed, "FF10 RAW")
# 4. Làm sạch tùy chọn
ff10_cleaned = clean_returns(
    ff10_parsed,
    clean=CLEAN,
    fill=FILL
)
# 5. Đổi sang lợi suất thập phân
ff10 = convert_to_decimal(ff10_cleaned)
# 6. Kiểm tra kết quả
ff10_report = inspect_returns(
    ff10,
    name="FF10 PROCESSED",
    decimal=True
)
display(ff10.head())


# FF48
ff48_raw = read_portfolio_file(
    PATH_FF48,
    n_assets=48
)

ff48_parsed = parse_returns(ff48_raw)

ff48_report_raw = inspect_returns(
    ff48_parsed, "FF48 RAW"
)

ff48_cleaned = clean_returns(
    ff48_parsed,
    clean=CLEAN,
    fill=FILL
)

ff48 = convert_to_decimal(ff48_cleaned)

ff48_report = inspect_returns(
    ff48,
    name="FF48 PROCESSED",
    decimal=True
)

display(ff48.head())

# FF100
ff100_raw = read_portfolio_file(
    PATH_FF100,
    n_assets=100
)

ff100_parsed = parse_returns(ff100_raw)

ff100_report_raw = inspect_returns(
    ff100_parsed, "FF100 RAW"
)

ff100_cleaned = clean_returns(
    ff100_parsed,
    clean=CLEAN,
    fill=FILL
)

ff100 = convert_to_decimal(ff100_cleaned)
ff100_report = inspect_returns(
    ff100,
    name="FF100 PROCESSED",
    decimal=True
)
display(ff100.head())


========== FF10 RAW ==========
dataset: FF10 RAW
n_months: 1203
n_assets: 10
start: 1926-07
end: 2026-09
missing_codes: 0
nan_cells: 0
inf_cells: 0
missing_months: 0

========== FF10 PROCESSED ==========
dataset: FF10 PROCESSED
n_months: 1203
n_assets: 10
start: 1926-07
end: 2026-09
missing_codes: 0
nan_cells: 0
inf_cells: 0
missing_months: 0


,Date,NoDur,Durbl,Manuf,Enrgy,HiTec,Telcm,Shops,Hlth,Utils,Other
0,1926-07-01,0.0144,0.1558,0.0470,-0.0114,0.0290,0.0083,0.0012,0.0185,0.0704,0.0214
1,1926-08-01,0.0399,0.0370,0.0280,0.0343,0.0266,0.0217,-0.0072,0.0417,-0.0170,0.0435
2,1926-09-01,0.0115,0.0498,0.0117,-0.0330,-0.0039,0.0242,0.0021,0.0069,0.0205,0.0031
3,1926-10-01,-0.0124,-0.0839,-0.0365,-0.0078,-0.0458,-0.0011,-0.0229,-0.0057,-0.0327,-0.0285
4,1926-11-01,0.0521,-0.0017,0.0427,0.0001,0.0471,0.0163,0.0645,0.0543,0.0440,0.0213



========== FF48 RAW ==========
dataset: FF48 RAW
n_months: 1203
n_assets: 48
start: 1926-07
end: 2026-09
missing_codes: 2400
nan_cells: 0
inf_cells: 0
missing_months: 0

========== FF48 PROCESSED ==========
dataset: FF48 PROCESSED
n_months: 1203
n_assets: 48
start: 1926-07
end: 2026-09
missing_codes: 0
nan_cells: 0
inf_cells: 0
missing_months: 0


,Date,Agric,Food,Soda,Beer,Smoke,Toys,Fun,Books,Hshld,...,Boxes,Trans,Whlsl,Rtail,Meals,Banks,Insur,RlEst,Fin,Other
0,1926-07-01,0.0236,0.0009,0.0,-0.0519,0.0129,0.0865,0.0249,0.5021,-0.0049,...,0.0770,0.0193,-0.2379,0.0008,0.0188,0.0461,-0.0054,0.0290,-0.0577,0.0520
1,1926-08-01,0.0223,0.0271,0.0,0.2703,0.0652,0.1681,-0.0076,0.4298,-0.0358,...,-0.0237,0.0485,0.0539,-0.0076,-0.0013,0.1071,0.0257,0.0531,0.0032,0.0676
2,1926-09-01,-0.0057,0.0159,0.0,0.0402,0.0126,0.0833,0.0648,-0.0491,0.0072,...,-0.0554,0.0010,-0.0787,0.0025,-0.0056,-0.0180,0.0072,-0.0306,-0.0483,-0.0358
3,1926-10-01,-0.0046,-0.0367,0.0,-0.0331,0.0107,-0.0140,-0.0509,0.0537,-0.0466,...,-0.0508,-0.0263,-0.1538,-0.0219,-0.0411,-0.1193,-0.0428,-0.0575,-0.0095,-0.0847
4,1926-11-01,0.0675,0.0626,0.0,0.0729,0.0456,0.0000,0.0183,-0.0662,-0.0053,...,0.0385,0.0161,0.0467,0.0654,0.0433,-0.0202,0.0358,0.0221,0.0513,0.0416



========== FF100 RAW ==========
dataset: FF100 RAW
n_months: 1203
n_assets: 100
start: 1926-07
end: 2026-09
missing_codes: 1455
nan_cells: 0
inf_cells: 0
missing_months: 0

========== FF100 PROCESSED ==========
dataset: FF100 PROCESSED
n_months: 1203
n_assets: 100
start: 1926-07
end: 2026-09
missing_codes: 0
nan_cells: 0
inf_cells: 0
missing_months: 0


,Date,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,ME1 BM5,ME1 BM6,ME1 BM7,ME1 BM8,ME1 BM9,...,BIG LoBM,ME10 BM2,ME10 BM3,ME10 BM4,ME10 BM5,ME10 BM6,ME10 BM7,ME10 BM8,ME10 BM9,BIG HiBM
0,1926-07-01,0.0,0.123656,0.0,-0.206349,-0.064864,-0.047429,-0.118628,0.157276,-0.016398,...,0.038451,0.011863,0.190166,0.014242,0.019459,0.024875,0.031859,0.003378,0.008576,0.0
1,1926-08-01,0.0,0.029904,0.0,0.080000,0.067495,-0.035639,0.032448,-0.016529,-0.053824,...,-0.000832,0.039828,0.085914,0.030099,0.011419,0.030283,0.060976,0.060606,0.112245,0.0
2,1926-09-01,0.0,-0.185830,0.0,-0.037037,-0.050512,-0.021739,-0.045714,-0.106099,0.003444,...,-0.003117,-0.019770,0.108102,0.008045,-0.009109,0.010956,-0.005436,-0.009524,-0.010703,0.0
3,1926-10-01,0.0,-0.041369,0.0,-0.115385,-0.053996,-0.177778,0.035554,-0.092233,-0.090684,...,-0.040564,-0.006422,-0.066177,-0.005326,-0.015745,-0.019279,-0.053544,-0.009615,-0.039246,0.0
4,1926-11-01,0.0,-0.082589,0.0,0.347826,0.012296,0.002027,-0.048789,-0.032010,-0.018459,...,0.067282,0.003788,0.004683,0.029192,0.011540,0.008811,0.036548,0.037217,0.032680,0.0


In [27]:
def make_window(
    df,
    test_start,
    train_months=60,
    test_months=12
):
    """
    Tạo một cửa sổ huấn luyện / kiểm thử.

    Input:
        df: dữ liệu đã đổi sang thập phân
        test_start: tháng bắt đầu kiểm thử

    Output:
        Dictionary chứa dữ liệu và tham số.
    """
    data = df.copy()

    data["Period"] = pd.PeriodIndex(
        data["Date"],
        freq="M"
    )

    data = data.set_index("Period")

    asset_cols = data.columns.drop("Date")

    test_start = pd.Period(
        test_start, freq="M"
    )

    train_start = test_start - train_months
    train_end = test_start - 1
    test_end = test_start + test_months - 1

    train_dates = pd.period_range(
        train_start, train_end, freq="M"
    )

    test_dates = pd.period_range(
        test_start, test_end, freq="M"
    )

    required = train_dates.append(test_dates)

    missing_dates = required.difference(data.index)

    if len(missing_dates):
        raise ValueError(
            f"Thiếu tháng: {missing_dates.tolist()}"
        )

    R_train = data.loc[
        train_dates, asset_cols
    ].to_numpy(dtype=float)

    R_test = data.loc[
        test_dates, asset_cols
    ].to_numpy(dtype=float)

    # Không để NaN/Inf âm thầm đi vào tối ưu
    if not np.isfinite(R_train).all():
        raise ValueError(
            f"R_train có NaN hoặc Inf: {test_start}"
        )

    if not np.isfinite(R_test).all():
        raise ValueError(
            f"R_test có NaN hoặc Inf: {test_start}"
        )

    # Ước lượng trên tập train
    mu = np.mean(R_train, axis=0)

    Sigma = np.cov(
        R_train,
        rowvar=False,
        ddof=1
    )

    # Danh mục phân bổ đều
    n_assets = len(asset_cols)
    x_equal = np.ones(n_assets) / n_assets

    # Lợi suất mục tiêu
    rho = float(mu @ x_equal)

    return {
        "train_start": str(train_start),
        "train_end": str(train_end),
        "test_start": str(test_start),
        "test_end": str(test_end),
        "R_train": R_train,
        "R_test": R_test,
        "mu": mu,
        "Sigma": Sigma,
        "rho": rho,
        "x_equal": x_equal,
        "asset_names": list(asset_cols),
        "n_assets": n_assets
    }


def build_rolling_windows(
    df,
    first_test="1976-07",
    last_test="2006-06",
    train_months=60,
    test_months=12
):
    """
    Tạo các cửa sổ kiểm thử nối tiếp nhau.
    last_test: tháng cuối cùng được kiểm thử.
    """
    first = pd.Period(first_test, freq="M")
    last = pd.Period(last_test, freq="M")

    windows = []
    current = first

    while current + test_months - 1 <= last:
        w = make_window(
            df,
            test_start=str(current),
            train_months=train_months,
            test_months=test_months
        )

        windows.append(w)
        current += test_months

    return windows


In [28]:
# FF10
windows_ff10 = build_rolling_windows(
    ff10,
    first_test=FIRST_TEST,
    last_test=LAST_TEST,
    train_months=TRAIN_MONTHS,
    test_months=TEST_MONTHS
)

# FF48
windows_ff48 = build_rolling_windows(
    ff48,
    first_test=FIRST_TEST,
    last_test=LAST_TEST,
    train_months=TRAIN_MONTHS,
    test_months=TEST_MONTHS
)

# FF100
windows_ff100 = build_rolling_windows(
    ff100,
    first_test=FIRST_TEST,
    last_test=LAST_TEST,
    train_months=TRAIN_MONTHS,
    test_months=TEST_MONTHS
)

print("FF10 windows:", len(windows_ff10))
print("FF48 windows:", len(windows_ff48))
print("FF100 windows:", len(windows_ff100))


FF10 windows: 30
FF48 windows: 30
FF100 windows: 30


In [33]:
w = windows_ff10[0]

print("===== F10 =====")
print("Huấn luyện:", w["train_start"], "->", w["train_end"])
print("Kiểm thử:", w["test_start"], "->", w["test_end"])

print("R_train:", w["R_train"].shape)
print("R_test:", w["R_test"].shape)
print("mu:", w["mu"].shape)
print("Sigma:", w["Sigma"].shape)
print("rho:", w["rho"])

w = windows_ff48[0]

print("\n ===== F48 =====")
print("Huấn luyện:", w["train_start"], "->", w["train_end"])
print("Kiểm thử:", w["test_start"], "->", w["test_end"])

print("R_train:", w["R_train"].shape)
print("R_test:", w["R_test"].shape)
print("mu:", w["mu"].shape)
print("Sigma:", w["Sigma"].shape)
print("rho:", w["rho"])

w = windows_ff100[0]

print("\n ===== F48 =====")
print("Huấn luyện:", w["train_start"], "->", w["train_end"])
print("Kiểm thử:", w["test_start"], "->", w["test_end"])

print("R_train:", w["R_train"].shape)
print("R_test:", w["R_test"].shape)
print("mu:", w["mu"].shape)
print("Sigma:", w["Sigma"].shape)
print("rho:", w["rho"])


===== F10 =====
Huấn luyện: 1971-07 -> 1976-06
Kiểm thử: 1976-07 -> 1977-06
R_train: (60, 10)
R_test: (12, 10)
mu: (10,)
Sigma: (10, 10)
rho: 0.00427

 ===== F48 =====
Huấn luyện: 1971-07 -> 1976-06
Kiểm thử: 1976-07 -> 1977-06
R_train: (60, 48)
R_test: (12, 48)
mu: (48,)
Sigma: (48, 48)
rho: 0.0037676736111111105

 ===== F48 =====
Huấn luyện: 1971-07 -> 1976-06
Kiểm thử: 1976-07 -> 1977-06
R_train: (60, 100)
R_test: (12, 100)
mu: (100,)
Sigma: (100, 100)
rho: 0.004785413166666666


### 1. Bài toán tối ưu danh mục đầu tư

Xét bài toán:

$$
\boxed{
\min_{x\in C}
\left\{
\frac1T\|\rho\mathbf1_T-Rx\|_2^2+\lambda\|x\|_1
\right\},
}
$$

trong đó:

- $R\in\mathbb R^{T\times n}$: ma trận lợi suất lịch sử.
- $x\in\mathbb R^n$: vector trọng số danh mục.
- $\widehat\mu\in\mathbb R^n$: vector lợi suất kỳ vọng.
- $\rho$: lợi suất mục tiêu.
- $\lambda>0$: tham số hiệu chỉnh.

Tập ràng buộc:

$$
\boxed{
C=\left\{
x\in\mathbb R^n:
\mathbf1_n^\top x=1,\;
\widehat\mu^\top x=\rho,\;
x\geq0
\right\}.
}
$$


Đặt:

$$
g(x)=\frac1T\|\rho\mathbf1_T-Rx\|_2^2,
\qquad
h(x)=\lambda\|x\|_1+\iota_C(x),
$$

trong đó $\iota_C$ là hàm chỉ của tập $C$.

Bài toán trở thành:

$$
\min_{x\in\mathbb R^n}\{g(x)+h(x)\}.
$$

Chọn $A=\partial h$ và $B=\nabla g$, ta có:

$$
\boxed{
A(x)=\lambda\partial\|x\|_1+N_C(x),
}
$$

$$
\boxed{
B(x)=\frac2T R^\top(Rx-\rho\mathbf1_T).
}
$$

Do đó, điều kiện tối ưu được viết dưới dạng:

$$
\boxed{0\in A(x^\ast)+B(x^\ast).}
$$

### 4. Hằng số Lipschitz

Với $B=\nabla g$, ta có:

$$
\|B(x)-B(y)\|_2\leq L\|x-y\|_2,
$$

trong đó:

$$
\boxed{
L=\frac2T\lambda_{\max}(R^\top R)
=\frac2T\|R\|_2^2.
}
$$

### 5. Toán tử giải và cỡ bước

Toán tử giải của $A$:

$$
J_{\gamma A}(z)
=\operatorname{prox}_{\gamma h}(z)
=\underset{x\in C}{\operatorname{argmin}}
\left\{
\frac12\|x-z\|_2^2+\gamma\lambda\|x\|_1
\right\}.
$$

Do $\|x\|_1=1$ với mọi $x\in C$, suy ra:

$$
\boxed{J_{\gamma A}(z)=P_C(z).}
$$

Cỡ bước của các thuật toán:

- **FBM:** $\gamma\in(0,2/L)$.
- **MFBSM:** $\gamma\in(0,1/L)$.
- **RFBSM:** $\lambda_k$ được cập nhật thích nghi.

### 6. Điểm khởi tạo

Chọn danh mục phân bổ đều:

$$
\boxed{
x^0=\left(\frac1n,\ldots,\frac1n\right)^\top.
}
$$

Điểm này thuộc $C$ khi $\rho=\widehat\mu^\top x^0$.